## Test of Adadetect

### Imports and Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Classifiers
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Your procedure
from procedure import AdaDetectERM, AdaDetectERMcv

In [2]:
def prepare_data(X, y, test_size=0.5):
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    X_test, X_null, y_test, y_null = train_test_split(
        X, y, test_size=test_size, random_state=42
    )

    return X_test, X_null, y_test, y_null

#### Classifiers

In [3]:
def get_classifiers():
    return {
        "knn": KNeighborsClassifier(3),
        "svm": SVC(probability=True),
        "decision_tree": DecisionTreeClassifier(max_depth=5),
        "random_forest": RandomForestClassifier(n_estimators=100),
        "mlp": MLPClassifier(max_iter=1000),
        "adaboost": AdaBoostClassifier(),
        "naive_bayes": GaussianNB(),
        "gaussian_process": GaussianProcessClassifier(1.0 * RBF(1.0)),
    }

In [4]:
def run_adadetect(model, x, xnull, level=0.1, use_cv=False):
    if use_cv:
        proc = AdaDetectERMcv(
            scoring_fn=model,
            cv_params=None  # optionally define grid
        )
    else:
        proc = AdaDetectERM(
            scoring_fn=model,
            split_size=0.5
        )

    rejection_set = proc.apply(x=x, level=level, xnull=xnull)

    return {
        "rejections": rejection_set,
        "n_rejections": len(rejection_set),
        "test_stats": proc.test_statistics,
        "null_stats": proc.null_statistics,
    }

In [5]:
def benchmark_classifiers(X, y, level=0.1):
    x, xnull = split_null_signal(X, y)

    results = {}
    classifiers = get_classifiers()

    for name, model in classifiers.items():
        print(f"Running: {name}")

        try:
            res = run_adadetect(model, x, xnull, level)
            results[name] = res["n_rejections"]
        except Exception as e:
            print(f"Failed: {name} -> {e}")
            results[name] = None

    return results

In [6]:
def plot_results(results):
    names = list(results.keys())
    values = [v if v is not None else 0 for v in results.values()]

    plt.figure()
    plt.bar(names, values)
    plt.xticks(rotation=45)
    plt.ylabel("Number of Rejections")
    plt.title("AdaDetect Classifier Comparison")
    plt.show()

In [7]:
from sklearn.inspection import DecisionBoundaryDisplay

def plot_decision_boundary(model, X, y):
    DecisionBoundaryDisplay.from_estimator(
        model, X, response_method="predict"
    )
    plt.scatter(X[:, 0], X[:, 1], c=y)
    plt.show()

In [8]:
# Example usage
results = benchmark_classifiers(X, y, level=0.1)
plot_results(results)

NameError: name 'X' is not defined